# 00 Environment Check

Validate the Fabric Spark runtime, Lakehouse attachment, project configuration files, and basic Delta table write/read behaviour.

## Initialise Run Context

This cell creates a unique run identifier and prints basic runtime metadata. When run in Fabric, it also prints the current workspace and default Lakehouse from NotebookUtils runtime context.

In [ ]:
# Cell purpose: Initialise Run Context.
from datetime import datetime, timezone
from pathlib import Path
import json
import uuid

run_id = str(uuid.uuid4())

print(f"run_id={run_id}")
print(f"utc_now={datetime.now(timezone.utc).isoformat()}")

# Fabric exposes workspace and default Lakehouse details through NotebookUtils.
# These values are unavailable in local Python kernels and can be absent in some Fabric contexts.
try:
    fabric_context = notebookutils.runtime.context
    print(f"workspace_name={fabric_context.get('currentWorkspaceName') or 'unknown'}")
    print(f"lakehouse_name={fabric_context.get('defaultLakehouseName') or 'none_attached'}")
except Exception as exc:
    print(f"fabric_context=unavailable ({type(exc).__name__})")

## Verify Spark Runtime

This cell confirms the notebook is running in a Fabric Spark session. It fails fast when opened in a local-only Python kernel.

In [ ]:
# Confirm Spark is available. This should pass in Fabric and fail clearly in local-only kernels.
try:
    spark_version = spark.version
    print(f"Spark available: {spark_version}")
except NameError as exc:
    raise RuntimeError("Spark session is not available. Run this notebook in Microsoft Fabric.") from exc

## Check Project Files

This cell checks whether the repository configuration files are available to the notebook runtime. If they are missing in Fabric, upload the repo files or package the Python project with the notebook.

In [ ]:
# Required config files should be published with the notebook or made available in the workspace files.
required_paths = [
    "config/sources.yml",
    "config/tables.yml",
    "config/dashboard_requirements.yml",
]

missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    print("Missing local notebook files:", missing)
    print("If running directly in Fabric, upload the repo files or package nem_fabric with the notebook.")
else:
    print("Config files found:", required_paths)

## Validate Lakehouse Table Access

This cell writes and reads a small Delta table. It proves that the notebook has a writable default Lakehouse table context before running ingestion jobs.

In [ ]:
# Write and read a tiny Delta table to prove the notebook has a writable Lakehouse table context.
health_table = "nem_environment_healthcheck"
payload = [{"run_id": run_id, "checked_at_utc": datetime.now(timezone.utc).isoformat(), "status": "ok"}]
df = spark.createDataFrame(payload)
df.write.format("delta").mode("append").saveAsTable(health_table)

display(spark.table(health_table).orderBy("checked_at_utc", ascending=False).limit(5))